# Consolidate database files from cohort searcher and from tabulation into one patients_database.csv

In [1]:
# define cohort

cohort = 'all_patients'

In [2]:
# read two csvs as dataframes

import pandas as pd

phi_df = pd.read_csv(f"/home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/{cohort}/Cardiac 3D Cine 4D Flow_with-HPI_updated-09-10-25.csv")
dei_df = pd.read_csv(f"/home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/{cohort}/Cardiac 3D Cine 4D Flow.csv", header=5)


In [3]:
# merge the two dataframes on the accession number column

dei_df['Accession Number'] = dei_df['Accession Number'].astype(str)
phi_df['Accession Number'] = phi_df['Accession Number'].astype(str)

merged_df = pd.merge(dei_df, phi_df, on='Accession Number', how='inner')
display(merged_df[['Accession Number', 'Phonetic ID_x','Phonetic ID_y', '3D Cine series', 'Series Descriptions']])


,Accession Number,Phonetic ID_x,Phonetic ID_y,3D Cine series,Series Descriptions
0,58657264,NaN,Pagoner,1100,"4ch, 2ch, 3ch, IVC and PAPVR, VSD, PAPVR, Ax S..."
1,R12588035,NaN,Jineca,800,"3ch, MR, 4ch, Sag 4D FLOW v350, Sag 4D FLOW v3..."
2,58811650,NaN,Coosimo,2300,"PA 2D Fast PC BH 250, PA 2D Fast PC BH 250, PA..."
3,58792366,Drusdinut,Drusdinut,2200,"3 Ch, Ax Stanford 4D Flow Heart V350, Ax Stanf..."
4,58516046,Phidecro,Phidecro,1000,"ORIG 3SL SAx T2Map BH, 3CH Cine STACK 6VPS HR,..."
...,...,...,...,...,...
262,56870969,Okarcug,Okarcug,1500,"sax strain, report-2023-10-21, rv3ch, 3ch, 4ch..."
263,56546881,Phunitul,Phunitul,900,"collaterals, PAs, asc ao, ao, [Loc:57.72] (1 S..."
264,56614718,Talodut,Talodut,1000,"[Loc:121.10] SAX MOLLI Post, MAG:SAX SSPS MDE ..."
265,56614412,Mijobost,Mijobost,900,"papvr, asd, [Loc:-50.98] SAX MOLLI Post, MAG:S..."


In [5]:
# dump to the working directory in this repo

merged_df.to_csv(f"/home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/{cohort}/patients_database.csv", index=False)

## QA: old patient database had only 180 patients and for some reason was missing patients that albert had already tabulated. identifying the missing patients to download from arterys here. no need to use this code otherwise. patient_database.csv creation is above

In [15]:
# Load the databases (your existing code)
old_patient_database = pd.read_csv(f"/home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/{cohort}/patients_database_depracated-09-10-25.csv")
new_patient_database = pd.read_csv(f"/home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/{cohort}/patients_database.csv")

# Check database sizes
print(f"Old database contains {len(old_patient_database)} patients")
print(f"New database contains {len(new_patient_database)} patients")

# Convert both accession number columns to strings and strip whitespace
old_accessions = set(old_patient_database['Accession Number'].dropna().astype(str).str.strip())
new_accessions = set(new_patient_database['Accession Number'].dropna().astype(str).str.strip())

print(f"Old database has {len(old_accessions)} unique accession numbers")
print(f"New database has {len(new_accessions)} unique accession numbers")

# Find overlap and new patients
overlap = old_accessions & new_accessions
new_patient_accessions = new_accessions - old_accessions

print(f"Overlapping patients: {len(overlap)}")
print(f"New patients: {len(new_patient_accessions)}")

# Get the new patients from the dataframe
if len(new_patient_accessions) > 0:
    # Convert accession numbers in dataframe to string for filtering
    new_patient_database_str = new_patient_database.copy()
    new_patient_database_str['Accession Number'] = new_patient_database_str['Accession Number'].astype(str).str.strip()
    
    new_patients = new_patient_database_str[new_patient_database_str['Accession Number'].isin(new_patient_accessions)]
    new_patients = new_patients.sort_values('Accession Number').reset_index(drop=True)
    
    print(f"\nFound {len(new_patients)} new patients:")
    print("=" * 80)
    
    # Display new patients with key information
    for idx, row in new_patients.iterrows():
        accession = row['Accession Number']
        
        # Try to get phonetic ID from different possible column names
        phonetic_id = "N/A"
        for col in ['Phonetic ID_x', 'Phonetic ID_y', 'Phonetic ID']:
            if col in row and pd.notna(row[col]):
                phonetic_id = row[col]
                break
                
        study_desc = row.get('Study Description', 'N/A')
        print(f"{idx+1:3d}. Accession: {accession:<15} | Phonetic ID: {phonetic_id:<10} | Study: {study_desc}")
    
    # Display the full dataframe of new patients
    print(f"\nDetailed view of all {len(new_patients)} new patients:")
    display(new_patients)
    
else:
    print("No new patients found!")

Old database contains 181 patients
New database contains 267 patients
Old database has 181 unique accession numbers
New database has 267 unique accession numbers
Overlapping patients: 180
New patients: 87

Found 87 new patients:
  1. Accession: 510445629       | Phonetic ID: Netstegup  | Study: MRI CARDIAC WO/W CONTRAST
  2. Accession: 56418896        | Phonetic ID: Quigliri   | Study: MRI CARDIAC WO/W CONTRAST
  3. Accession: 56427478        | Phonetic ID: Stapimdem  | Study: MRI CARDIAC WO/W CONTRAST
  4. Accession: 56614412        | Phonetic ID: Mijobost   | Study: MRI CARDIAC WO/W CONTRAST
  5. Accession: 56627191        | Phonetic ID: Amifer     | Study: MRI CARDIAC WO/W CONTRAST
  6. Accession: 56647991        | Phonetic ID: Aruborn    | Study: MRI CARDIAC WO/W CONTRAST
  7. Accession: 56740527        | Phonetic ID: Prubodusk  | Study: MRI CARDIAC WO/W CONTRAST
  8. Accession: 56755075        | Phonetic ID: Lurakouk   | Study: MRI CARDIAC WO/W CONTRAST
  9. Accession: 56786189   

,Accession Number,Phonetic ID_x,Include,3D Cine series,3D Cine Quality,4d flow,4D Flow Quality,4D Flow archive (Stanford),Study Key,Phonetic ID_y,...,MagneticFieldStrengths,ImageOrientationPatient,ImagePositionPatient,SpacingBetweenSlices,Slice thickness,PixelSpacing,Window Center,WindowWidth,HeartRate,Additional History
0,510445629,NaN,yes,800,ok,yes,NaN,ss104c/Exam4432-Series10/ScanArchive_858657MR4...,2.25.23089462276356203754936174979414892177,Netstegup,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,56418896,Quigliri,yes,1700,ok,yes,NaN,KOPMR1/Exam7109-Series18/ScanArchive_858657MR5...,2.25.256760205675083972347504607222570381007,Quigliri,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,56427478,Stapimdem,yes,900,ok,yes,NaN,ss104c/Exam1188-Series11/ScanArchive_858657MR4...,2.25.89611323338608189869224176978231421046,Stapimdem,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,56614412,Mijobost,yes,900,ok,yes,NaN,ss104c/Exam874-Series11/ScanArchive_858657MR4_...,2.25.296417948121325210588096742503929045869,Mijobost,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,56627191,Amifer,yes,900,ok,yes,NaN,ss104c/Exam1238-Series11/ScanArchive_858657MR4...,2.25.244199229655107535954432555184941109938,Amifer,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82,58763262,NaN,yes,800,ok,yes,NaN,ss104c/Exam4538-Series10/ScanArchive_858657MR4...,2.25.338829831853330552812936538866563665140,Klusumay,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
83,58790675,NaN,yes,2000,ok,yes,NaN,KOPMR2/Exam15162-Series21/ScanArchive_858657MR...,2.25.42715556949060168097695599975915048270,Stragotom,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
84,58791293,NaN,yes,2000,ok,yes,NaN,KOPMR2/Exam14968-Series21/ScanArchive_858657MR...,2.25.38940618701754679874718556117474521664,Diboscey,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
85,58811650,NaN,yes,2300,ok,yes,NaN,KOPMR1/Exam18926-Series25/ScanArchive_858657MR...,2.25.195349058639959854169562786426680773164,Coosimo,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [21]:
#print a list of the phonetic ids of the new patients but without apostrophes
new_patients_list = new_patients['Phonetic ID_y'].astype(str).tolist()
new_patients_list = [id.replace("'", "") for id in new_patients_list]
print(f"length of new patients list: {len(new_patients_list)}")
print(new_patients_list)

length of new patients list: 87
['Netstegup', 'Quigliri', 'Stapimdem', 'Mijobost', 'Amifer', 'Aruborn', 'Prubodusk', 'Lurakouk', 'Detodu', 'Gristropu', 'Rarunig', 'Quapomook', 'Hueregour', 'Vokuba', 'Okarcug', 'Wetufru', 'Gascoyar', 'Sepigoo', 'Gikolu', 'Swibihek', 'Fyenpip', 'Rirurask', 'Nedrikjut', 'Gukedoo', 'Ebukhot', 'Jacibkut', 'Tisupey', 'Tosbeybrof', 'Masobo', 'Jienekis', 'Keniso', 'Asonlig', 'Nafepim', 'Bitistep', 'Rotomouk', 'Ceyebum', 'Krujouque', 'Gifranu', 'Souloochap', 'Sigurue', 'Lamoga', 'Kufootag', 'Fohako', 'Pugishof', 'Diepami', 'Naniduf', 'Gepeta', 'Balboloop', 'Sopeyak', 'Janiru', 'Tomamel', 'Liepoja', 'Gueshifa', 'Bogeebo', 'Uxodir', 'Befeto', 'Hisrokock', 'Eposur', 'Dudoblo', 'Elagieg', 'Nafenot', 'Lumutot', 'Tupulou', 'Konodin', 'Emalem', 'Obdufar', 'Sulokuf', 'Quequopli', 'Nobogik', 'Sevoopir', 'Hapamiem', 'Gidusi', 'Tecerouf', 'Kemotub', 'Burapo', 'Dublafer', 'Secochi', 'Boudubat', 'Pagoner', 'Quidesue', 'Liegibla', 'Soyefaf', 'Klusumay', 'Stragotom', 'Dibosce

In [22]:
copyable_list = ""
for id in new_patients_list:
    copyable_list += f"{id}, "
print(copyable_list)

Netstegup, Quigliri, Stapimdem, Mijobost, Amifer, Aruborn, Prubodusk, Lurakouk, Detodu, Gristropu, Rarunig, Quapomook, Hueregour, Vokuba, Okarcug, Wetufru, Gascoyar, Sepigoo, Gikolu, Swibihek, Fyenpip, Rirurask, Nedrikjut, Gukedoo, Ebukhot, Jacibkut, Tisupey, Tosbeybrof, Masobo, Jienekis, Keniso, Asonlig, Nafepim, Bitistep, Rotomouk, Ceyebum, Krujouque, Gifranu, Souloochap, Sigurue, Lamoga, Kufootag, Fohako, Pugishof, Diepami, Naniduf, Gepeta, Balboloop, Sopeyak, Janiru, Tomamel, Liepoja, Gueshifa, Bogeebo, Uxodir, Befeto, Hisrokock, Eposur, Dudoblo, Elagieg, Nafenot, Lumutot, Tupulou, Konodin, Emalem, Obdufar, Sulokuf, Quequopli, Nobogik, Sevoopir, Hapamiem, Gidusi, Tecerouf, Kemotub, Burapo, Dublafer, Secochi, Boudubat, Pagoner, Quidesue, Liegibla, Soyefaf, Klusumay, Stragotom, Diboscey, Coosimo, Jineca, 
